# 02 · Exploración descriptiva de cinco años

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

In [ ]:
from src.stability import EDA_DIR, run_eda_five_years

eda = run_eda_five_years(cfg)
m5 = eda["matrix_five_years"]
pd.crosstab(m5["abc_class"], m5["xyz_class"].fillna("-"), margins=True)

Acuerdo y PSI de cada año frente al anterior y frente a la matriz de cinco años (descriptivo).

In [ ]:
eda["stability_descriptive"]

Distribuciones por artículo: proporción de ceros en días activos, media frente a mediana, gasto y CV.

In [ ]:
describe = eda["describe"]
describe.describe()

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
active = m5[m5["exclusion_reason"] != "inactive_in_set"]
axes[0, 0].plot(np.arange(1, len(active) + 1), active["cum_spend"] / active["spend"].sum())
axes[0, 0].set(title="Curva ABC (5 años)", xlabel="artículos ordenados por gasto", ylabel="gasto acumulado")
axes[0, 1].hist(m5["cv"].dropna(), bins=50)
axes[0, 1].set(title="CV de unidades (%)", xlabel="CV")
axes[1, 0].hist(describe["zero_share"].dropna(), bins=50)
axes[1, 0].set(title="Proporción de ceros (días activos)")
axes[1, 1].scatter(describe["median_daily_units"], describe["mean_daily_units"], s=4)
axes[1, 1].set(title="Media frente a mediana diaria", xlabel="mediana", ylabel="media")
fig.tight_layout()
fig.savefig(Path(EDA_DIR) / "distributions.png", dpi=120)
fig